In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore 
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
loader = PyPDFLoader("../data/health_checkup_report.pdf")
docs = loader.load()

In [5]:
len(docs)

4

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitter_docs = splitter.split_documents(docs)

In [7]:
len(splitter_docs)

7

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 
vector_store = InMemoryVectorStore.from_documents(
    documents=splitter_docs,
    embedding=embeddings
)

### Agent = Tool, LLM, prompt

In [9]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retriever the relevant data of the pdf Documents, and these documents have details about medical reports.
    """
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [10]:
llm = ChatOpenAI(model = "gpt-5")

In [11]:
System_Prompt = """
    You are a helpful assistent that answer question using retrieved context.
    Always use the `retriever_tool` tool for question requiring external knowladge

"""

In [13]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [ ]:
query = "What is the name of patient, and what is the name of doctors "
response = agent.invoke({"messages": [{"role":"user", "content":query}]})

In [16]:
result = response["messages"][-1].content

In [17]:
print(result)

{
  "medications": {
    "status": "not_found",
    "details": "No Medications/Rx/Prescription section was found in the provided PDF excerpts."
  },
  "diagnosis": {
    "status": "not_found",
    "details": "No Diagnosis/Impression section was found in the provided PDF excerpts."
  },
  "summary": {
    "document_note": "Synthetic educational dataset; not for clinical use.",
    "overall_status": "Sample values are mostly within the example reference ranges.",
    "flagged": [
      {
        "test": "LDL cholesterol",
        "result": "106 mg/dL",
        "comment": "Slightly above the displayed optimal target of <100 mg/dL."
      }
    ],
    "diabetes_example": {
      "fasting_glucose": "96 mg/dL",
      "hba1c": "5.6%",
      "interpretation": "Within non-diabetic ranges (per document metadata)."
    },
    "kidney_example": {
      "creatinine": "0.86 mg/dL",
      "eGFR": "108 mL/min/1.73m²",
      "interpretation": "Within displayed example ranges (per document metadata)."
 